# Experiment: TrustSignal EZKL Experiments

Objective:
- Calibrate zkML settings per vertical (deed, healthcare, KYC).
- Track proof generation time vs accuracy tradeoffs.
- Track witness size and SRS parameters per circuit build.


In [ ]:
from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path
from statistics import fmean

NOTEBOOK_ROOT = Path.cwd()
ARTIFACT_DIR = NOTEBOOK_ROOT / "notebooks" / "artifacts" / "ezkl"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

try:
    import ezkl  # type: ignore
    EZKL_AVAILABLE = True
except Exception:
    ezkl = None
    EZKL_AVAILABLE = False

{
    "cwd": str(NOTEBOOK_ROOT),
    "artifact_dir": str(ARTIFACT_DIR),
    "ezkl_available": EZKL_AVAILABLE,
}


## Plan

- Hypothesis: each vertical requires different calibration/SRS settings for acceptable proof latency.
- Variables to sweep: scale, lookup range, decomposition settings, and SRS logrows (`k`).
- Metrics to record: calibration output fields, `proof_time_ms`, model quality metric, witness bytes, and SRS parameters.


In [ ]:
vertical_registry = {
    "deed": {
        "model_input_shape": [1, 512],
        "model_output_shape": [1, 4],
        "quality_metric": "fraud_auc",
        "target_quality": 0.97,
    },
    "healthcare": {
        "model_input_shape": [1, 768],
        "model_output_shape": [1, 6],
        "quality_metric": "credential_f1",
        "target_quality": 0.94,
    },
    "kyc": {
        "model_input_shape": [1, 384],
        "model_output_shape": [1, 3],
        "quality_metric": "risk_precision",
        "target_quality": 0.95,
    },
}

vertical_registry


## Calibration Output Tracker

If EZKL is available in this environment, replace placeholders with real outputs from `ezkl.calibrate_settings()`.
Otherwise, keep this as a versioned experiment ledger.


In [ ]:
calibration_runs = [
    {
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "vertical": "deed",
        "model_input_shape": vertical_registry["deed"]["model_input_shape"],
        "model_output_shape": vertical_registry["deed"]["model_output_shape"],
        "calibration_output": {
            "input_scale": 12,
            "param_scale": 12,
            "lookup_range": [0, 4096],
            "logrows": 17,
            "status": "placeholder",
        },
    },
    {
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "vertical": "healthcare",
        "model_input_shape": vertical_registry["healthcare"]["model_input_shape"],
        "model_output_shape": vertical_registry["healthcare"]["model_output_shape"],
        "calibration_output": {
            "input_scale": 13,
            "param_scale": 13,
            "lookup_range": [0, 8192],
            "logrows": 18,
            "status": "placeholder",
        },
    },
    {
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "vertical": "kyc",
        "model_input_shape": vertical_registry["kyc"]["model_input_shape"],
        "model_output_shape": vertical_registry["kyc"]["model_output_shape"],
        "calibration_output": {
            "input_scale": 11,
            "param_scale": 11,
            "lookup_range": [0, 2048],
            "logrows": 16,
            "status": "placeholder",
        },
    },
]

for row in calibration_runs:
    print(row["vertical"], row["calibration_output"])


## Proof Time vs Accuracy Tradeoff


In [ ]:
proof_accuracy_runs = [
    {"vertical": "deed", "experiment": "deed-k16", "proof_time_ms": 1240, "quality": 0.962},
    {"vertical": "deed", "experiment": "deed-k17", "proof_time_ms": 1475, "quality": 0.973},
    {"vertical": "healthcare", "experiment": "health-k17", "proof_time_ms": 1688, "quality": 0.931},
    {"vertical": "healthcare", "experiment": "health-k18", "proof_time_ms": 2021, "quality": 0.947},
    {"vertical": "kyc", "experiment": "kyc-k15", "proof_time_ms": 930, "quality": 0.941},
    {"vertical": "kyc", "experiment": "kyc-k16", "proof_time_ms": 1114, "quality": 0.956},
]

by_vertical = {}
for v in vertical_registry:
    rows = [r for r in proof_accuracy_runs if r["vertical"] == v]
    if not rows:
        continue
    best_quality = max(rows, key=lambda r: r["quality"])
    fastest = min(rows, key=lambda r: r["proof_time_ms"])
    by_vertical[v] = {
        "fastest": fastest,
        "best_quality": best_quality,
        "avg_proof_time_ms": round(fmean([r["proof_time_ms"] for r in rows]), 2),
    }

by_vertical


## Witness Size + SRS Parameter Tracker


In [ ]:
witness_srs_runs = [
    {"vertical": "deed", "circuit_id": "deed-v2", "witness_bytes": 2_410_000, "srs_k": 17, "srs_params": "kzg_bn254"},
    {"vertical": "healthcare", "circuit_id": "health-v1", "witness_bytes": 3_020_000, "srs_k": 18, "srs_params": "kzg_bn254"},
    {"vertical": "kyc", "circuit_id": "kyc-v3", "witness_bytes": 1_880_000, "srs_k": 16, "srs_params": "kzg_bn254"},
]

# Quick sanity checks
for row in witness_srs_runs:
    if row["srs_k"] < 15:
        raise ValueError(f"Unexpectedly low srs_k for {row['vertical']}: {row['srs_k']}")

witness_srs_runs


## Persist Experiment Snapshot


In [ ]:
snapshot = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "ezkl_available": EZKL_AVAILABLE,
    "vertical_registry": vertical_registry,
    "calibration_runs": calibration_runs,
    "proof_accuracy_runs": proof_accuracy_runs,
    "witness_srs_runs": witness_srs_runs,
}

snapshot_path = ARTIFACT_DIR / f"ezkl-snapshot-{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.json"
snapshot_path.write_text(json.dumps(snapshot, indent=2))
str(snapshot_path)


## Results

- Key observations:
  - Fill after running real calibrations and proof jobs.
- Surprises or failure modes:
  - Note any vertical where proof latency rises sharply with marginal quality gain.
- Decision:
  - Promote settings that meet quality targets with lowest proof latency and smallest witness.


In [ ]:
result = {
    "quality_targets": {v: cfg["target_quality"] for v, cfg in vertical_registry.items()},
    "current_best_by_vertical": {
        v: {
            "experiment": by_vertical[v]["best_quality"]["experiment"],
            "quality": by_vertical[v]["best_quality"]["quality"],
            "proof_time_ms": by_vertical[v]["best_quality"]["proof_time_ms"],
        }
        for v in by_vertical
    },
}
result


## Next Steps

- Replace placeholder calibration rows with real `ezkl.calibrate_settings()` outputs.
- Add automated extraction of witness byte size from generated witness files.
- Add charting for quality vs latency over time per vertical.
